# LangChain, RAG, Vector Databases, PostgreSQL & MongoDB — Revision Notes

---

# 1. Why LangChain Exists

Before LangChain, building an LLM application looked like:

```python
text = load_pdf()
chunks = split_text(text)
vectors = embed(chunks)
db.store(vectors)

query_vector = embed(question)
docs = db.search(query_vector)

prompt = build_prompt(docs, question)
answer = llm(prompt)
```

Every developer was repeatedly solving the same problems:

* Load documents
* Split documents
* Generate embeddings
* Store embeddings
* Retrieve relevant chunks
* Build prompts
* Parse outputs
* Call tools
* Add memory

LangChain's goal:

> Standardize common LLM application components so they can be composed together easily.

---

# 2. Runnables

## Problem Before Runnables

Earlier LangChain had many chain classes:

* LLMChain
* RetrievalQA
* ConversationalRetrievalChain
* etc.

Too many specialized abstractions.

---

## Runnable Concept

A Runnable is:

> A reusable unit of execution.

Examples:

* Prompt
* LLM
* Retriever
* Output Parser
* Custom Function

can all become Runnables.

---

## LCEL

```python
chain = prompt | model | parser
```

Equivalent to:

```text
Input
  ↓
Prompt
  ↓
Model
  ↓
Parser
  ↓
Output
```

---

## Why Runnables Matter

Every Runnable automatically supports:

```python
.invoke()
.batch()
.stream()
.ainvoke()
```

This gives:

* Composability
* Streaming
* Async execution
* Tracing
* Reusability

---

## Trade-Off

Advantages:

* Very composable
* Easy chaining
* Standard interface

Disadvantages:

* Can feel abstract
* Execution flow is less obvious for beginners

---

# 3. Document Loaders

## Purpose

Document Loaders bring external data into LangChain.

Examples:

* PDFs
* Websites
* Notion
* Slack
* Word files
* CSVs

---

## Standard Output

Everything becomes:

```python
Document(
    page_content="...",
    metadata={}
)
```

---

## Why This Matters

Without a standard format:

```python
PDF -> list
Website -> HTML
Slack -> JSON
```

Every downstream component becomes messy.

Document Loader solves this.

---

## Important Insight

Bad document loading = Bad RAG.

Things to evaluate:

* Table extraction
* OCR support
* Metadata quality
* Page preservation
* Header/footer removal

---

# 4. Text Splitters

## Problem

LLMs cannot consume huge documents efficiently.

Example:

100-page RFQ

Embedding the entire RFQ as one vector is usually bad.

---

## Goal

Break documents into meaningful chunks.

---

## Common Splitter

```python
RecursiveCharacterTextSplitter
```

Tries:

```text
Paragraph
 ↓
Line
 ↓
Sentence
 ↓
Word
 ↓
Character
```

---

## Chunk Size Trade-Off

### Small Chunks

Pros:

* Precise retrieval
* Less noise

Cons:

* Less context

---

### Large Chunks

Pros:

* More context

Cons:

* Retrieval becomes less precise

---

## Chunk Overlap

Example:

```python
chunk_size=1000
chunk_overlap=150
```

Purpose:

Prevent information loss at chunk boundaries.

---

## Key Insight

For business documents:

Section-aware chunking > Fixed-size chunking

Example:

```text
Eligibility
Budget
Timeline
Requirements
```

should ideally stay separate.

---

# 5. Embeddings

Embedding Model converts text into vectors.

Example:

```text
"The payment terms are 30 days"
```

becomes:

```python
[0.21, -0.45, 0.98, ...]
```

Hundreds or thousands of dimensions.

---

## Goal

Capture semantic meaning.

Similar meanings should produce nearby vectors.

---

# 6. Vector Databases

## Most Important Insight

Vector DBs do NOT exist because vectors are difficult to store.

Vectors are just numbers.

PostgreSQL can store them.

MongoDB can store them.

Files can store them.

Storage is NOT the problem.

---

## Real Problem

Search.

Given:

```text
1 million vectors
```

Find:

```text
Most similar vectors
```

efficiently.

---

## Traditional Database Search

Good at:

```text
ID = 123
Age > 25
Salary < 100000
```

---

## Vector Search

Asks:

```text
Which vectors are closest in meaning?
```

This is a nearest-neighbor search problem.

---

## Why Not Brute Force?

Without vector indexing:

```text
Compare query
against every vector.
```

Complexity:

```text
O(N)
```

Too slow at scale.

---

## Vector DB Purpose

Optimize:

```text
Nearest Neighbor Search
```

using:

* HNSW
* IVF
* PQ

etc.

---

## Key Idea

Vector DBs trade:

```text
Perfect accuracy
```

for

```text
Massive speed
```

using Approximate Nearest Neighbor Search (ANN).

---

# 7. PostgreSQL vs MongoDB vs Vector DB

---

## PostgreSQL

Optimizes for:

```text
Correctness
Consistency
Relationships
Transactions
```

Best for:

* Banking
* Payments
* Orders
* Inventory
* Financial systems

---

## MongoDB

Optimizes for:

```text
Flexibility
Developer speed
Schema evolution
```

Best for:

* User profiles
* CMS systems
* Content platforms
* Social media metadata

---

## Vector Databases

Optimizes for:

```text
Semantic Search
Similarity Search
Retrieval
Embeddings
```

Best for:

* RAG
* Recommendations
* Knowledge Bases
* Semantic Search

---

# 8. ACID

ACID means:

---

## Atomicity

Either:

```text
Everything succeeds
```

or

```text
Nothing succeeds
```

---

## Consistency

Rules remain valid.

Example:

```text
Balance cannot become invalid.
```

---

## Isolation

Concurrent operations do not corrupt each other.

---

## Durability

Committed data survives crashes.

---

# 9. Does PostgreSQL Support ACID?

Yes.

Postgres was designed around ACID from the beginning.

---

# 10. Does MongoDB Support ACID?

Yes.

Modern MongoDB supports:

```text
Multi-document transactions
```

with ACID guarantees.

Important:

MongoDB is NOT "No ACID."

That is an outdated statement.

---

# 11. Then Why Is PostgreSQL Still Preferred for Finance?

Because PostgreSQL's philosophy is:

```text
Data integrity first
```

and it has decades of maturity in transactional workloads.

---

# 12. Do Vector Databases Support ACID?

Depends on the database.

Some support stronger guarantees than others.

However:

Vector DBs are not primarily optimized for:

```text
Financial correctness
```

They are optimized for:

```text
Similarity search
Low latency retrieval
High-dimensional indexing
```

---

# 13. Architect's Mental Model

Do NOT ask:

```text
Should I use SQL or NoSQL?
```

Ask:

```text
What queries dominate?

What guarantees do I need?

What happens if data becomes inconsistent?

What relationships exist?

What scale am I expect?
```

---

# 14. The Three Types of Data

## Truth

Examples:

```text
Money
Orders
Payments
Inventory
```

Use:

```text
PostgreSQL
MySQL
```

---

## Content

Examples:

```text
Profiles
Posts
Comments
CMS Data
```

Use:

```text
MongoDB
```

---

## Meaning

Examples:

```text
Embeddings
RAG
Recommendations
Similarity Search
```

Use:

```text
Qdrant
Pinecone
Weaviate
FAISS
pgvector
```

---

# Final Takeaway

Every database is optimizing for something different.

PostgreSQL:

```text
Transactional correctness
```

MongoDB:

```text
Flexible document storage
```

Vector Database:

```text
Semantic similarity search
```

The best engineers do not start by choosing a database.

They start by understanding:

```text
Data
Queries
Access Patterns
Consistency Requirements
Scale
```

Then they select the database that optimizes for those requirements.

```
```
